# CA4: Fine-tuning PaliGemma on CLEVR-COGEN-A

This notebook trains a PaliGemma-3B vision-language model on the CLEVR-COGEN-A counting task using LoRA + 8-bit quantization. The flow follows the course rules: overview → config (imports, seeds, hyperparameters) → data → model → training/evaluation → qualitative samples.

In [ ]:
import os
import random
import json
import logging
from dataclasses import dataclass
from typing import Any, Dict, List

import numpy as np
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import (
    BitsAndBytesConfig,
    PaliGemmaForConditionalGeneration,
    PaliGemmaProcessor,
    Trainer,
    TrainingArguments,
)
import evaluate

# Logging and determinism
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Device: {device}")

# Centralized configuration
config = {
    "model_id": os.getenv("MODEL_ID", "google/paligemma-3b-mix-224"),
    "dataset_split": "train[:20%]",  # 20% subset per assignment
    "test_size": 0.1,
    "max_length": 300,
    "train_batch_size": 4,
    "eval_batch_size": 4,
    "grad_accum": 4,
    "learning_rate": 1e-4,
    "num_epochs": 1,
    "eval_steps": 100,
    "save_steps": 100,
    "logging_steps": 10,
    "lora_rank": 64,
    "lora_alpha": 64,
    "lora_dropout": 0.05,
    "output_dir": "./paligemma-clevr-finetuned",
}

# Optional smoke test: set MAX_STEPS env var (e.g., 20) to cap training
SMOKE_MAX_STEPS = os.getenv("MAX_STEPS")
MAX_STEPS = int(SMOKE_MAX_STEPS) if SMOKE_MAX_STEPS else -1
RUN_TRAINING = os.getenv("RUN_TRAINING", "false").lower() in {"1", "true", "yes"}
logger.info(f"Run training: {RUN_TRAINING}; max_steps={MAX_STEPS}")


## Data Loading and Preprocessing

Loads the CLEVR-COGEN-A subset, applies a 90/10 train/validation split, and prepares inputs for PaliGemma. Text prompts are prefixed with `<image>` as required by the processor. Padding tokens are masked with `-100` so they are ignored in the loss.

In [ ]:
# Load dataset (20% subset) and split
raw_dataset = load_dataset("leonardPKU/clevr_cogen_a_train", split=config["dataset_split"])
split_datasets = raw_dataset.train_test_split(test_size=config["test_size"], seed=SEED)
train_dataset = split_datasets["train"]
val_dataset = split_datasets["test"]
logger.info(f"Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}")

# Processor
processor = PaliGemmaProcessor.from_pretrained(config["model_id"])


def preprocess_function(batch: Dict[str, Any]) -> Dict[str, Any]:
    questions = batch["problem"]
    images = batch["image"]
    answers = batch["solution"]

    processed_images = []
    prompts = []
    targets = []
    for q, img, ans in zip(questions, images, answers):
        try:
            img = img.convert("RGB").resize((224, 224))
            processed_images.append(img)
            prompts.append(f"<image> {q}")
            targets.append(str(ans))
        except Exception as exc:  # skip invalid samples
            logger.warning("Skipping sample due to image issue: %s", exc)

    if not processed_images:
        return {}

    encoder_inputs = processor(
        images=processed_images,
        text=prompts,
        padding="max_length",
        truncation=True,
        max_length=config["max_length"],
        return_tensors="pt",
    )

    decoder_inputs = processor.tokenizer(
        text_target=targets,
        padding="max_length",
        truncation=True,
        max_length=config["max_length"],
        return_tensors="pt",
    )

    labels = decoder_inputs["input_ids"].clone()
    labels[decoder_inputs["attention_mask"] == 0] = -100
    encoder_inputs["labels"] = labels
    return encoder_inputs


processed_train_dataset = train_dataset.map(
    preprocess_function, batched=True, remove_columns=train_dataset.column_names
)
processed_val_dataset = val_dataset.map(
    preprocess_function, batched=True, remove_columns=val_dataset.column_names
)

processed_train_dataset.set_format(type="torch")
processed_val_dataset.set_format(type="torch")
logger.info("Preprocessing complete")


## Model, Quantization, and LoRA Configuration

Loads the PaliGemma model with 8-bit quantization (BitsAndBytes) and applies LoRA adapters to attention and MLP projections. Only ~3% of parameters are trainable.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
)

model = PaliGemmaForConditionalGeneration.from_pretrained(
    config["model_id"],
    device_map="auto",
    quantization_config=bnb_config,
)

lora_config = LoraConfig(
    r=config["lora_rank"],
    lora_alpha=config["lora_alpha"],
    lora_dropout=config["lora_dropout"],
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.config.use_cache = False  # required for training with gradient checkpointing if enabled

# Parameter stats
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
logger.info(f"Trainable params: {trainable/1e6:.1f}M ({100*trainable/total:.2f}%)")


## Training Configuration and Metrics

Defines Trainer setup, ROUGE metrics, and optional smoke-test knob via `MAX_STEPS`. Set `RUN_TRAINING=True` (env var) to execute full training.

In [ ]:
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    labels = np.where(labels != -100, labels, processor.tokenizer.pad_token_id)
    decoded_preds = processor.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = processor.batch_decode(labels, skip_special_tokens=True)
    scores = rouge.compute(predictions=decoded_preds, references=decoded_labels)
    return {
        "rouge1": scores["rouge1"],
        "rouge2": scores["rouge2"],
        "rougeL": scores["rougeL"],
    }

training_args = TrainingArguments(
    output_dir=config["output_dir"],
    num_train_epochs=config["num_epochs"],
    per_device_train_batch_size=config["train_batch_size"],
    per_device_eval_batch_size=config["eval_batch_size"],
    gradient_accumulation_steps=config["grad_accum"],
    evaluation_strategy="steps",
    eval_steps=config["eval_steps"],
    save_strategy="steps",
    save_steps=config["save_steps"],
    logging_steps=config["logging_steps"],
    learning_rate=config["learning_rate"],
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    remove_unused_columns=False,
    fp16=torch.cuda.is_available(),
    max_steps=MAX_STEPS,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=processed_train_dataset,
    eval_dataset=processed_val_dataset,
    tokenizer=processor.tokenizer,
    compute_metrics=compute_metrics,
)


## Run Training / Evaluation

Set `RUN_TRAINING=True` (environment variable) for a full run. For a quick smoke test, set `MAX_STEPS` (e.g., 20).

In [ ]:
if RUN_TRAINING:
    logger.info("Starting training...")
    train_result = trainer.train()
    trainer.save_model(config["output_dir"])
    metrics = trainer.evaluate()
    logger.info("Eval metrics: %s", json.dumps(metrics, indent=2))
else:
    logger.info("Training skipped. Set RUN_TRAINING=True to train. For smoke tests set MAX_STEPS.")


## Qualitative Check

Utility to generate a prediction for one validation sample.

In [ ]:
def generate_answer(sample_id: int = 0) -> Dict[str, str]:
    sample = val_dataset[sample_id]
    image = sample["image"].convert("RGB").resize((224, 224))
    prompt = f"<image> {sample['problem']}"
    inputs = processor(text=prompt, images=image, return_tensors="pt").to(device)
    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=10)
    prediction = processor.batch_decode(outputs, skip_special_tokens=True)[0]
    return {
        "question": sample["problem"],
        "ground_truth": str(sample["solution"]),
        "prediction": prediction,
    }

# Example usage (uncomment to run after training)
# generate_answer(0)
